# Vectors and Spaces

Companion notebook for the [Vectors and Spaces](https://ml-viz.vercel.app/courses/linear-algebra/01-vectors-and-spaces) lesson.

We'll build intuition for vectors, norms, dot products, and orthogonality using NumPy and Matplotlib.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

plt.style.use('dark_background')
plt.rcParams.update({
    'figure.facecolor': '#0f1117',
    'axes.facecolor':   '#1a1d27',
    'axes.edgecolor':   '#30344a',
    'axes.labelcolor':  '#e2e8f0',
    'xtick.color':      '#94a3b8',
    'ytick.color':      '#94a3b8',
    'text.color':       '#e2e8f0',
    'grid.color':       '#30344a',
})

## Vector operations

In [ ]:
u = np.array([2.0, 1.0])
v = np.array([1.0, 3.0])

print('u + v =', u + v)
print('2 * u =', 2 * u)
print('dot product u·v =', np.dot(u, v))
print('‖u‖ =', np.linalg.norm(u))
print('‖v‖ =', np.linalg.norm(v))

cos_theta = np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v))
angle_deg = np.degrees(np.arccos(np.clip(cos_theta, -1, 1)))
print(f'Angle between u and v: {angle_deg:.1f}°')

## Visualizing vectors and their angle

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: vectors in 2D space
ax = axes[0]
origin = np.array([0, 0])
colors = ['#6366f1', '#2dd4bf', '#f97316']
vectors = [u, v, u + v]
labels = ['u = [2,1]', 'v = [1,3]', 'u+v']

for vec, color, label in zip(vectors, colors, labels):
    ax.annotate('', xy=vec, xytext=origin,
                arrowprops=dict(arrowstyle='->', color=color, lw=2))
    ax.text(vec[0] * 1.05, vec[1] * 1.05, label, color=color, fontsize=11)

# Parallelogram for addition
para = plt.Polygon([origin, u, u+v, v], fill=False,
                   edgecolor='#6366f1', alpha=0.3, linestyle='--')
ax.add_patch(para)
ax.set_xlim(-0.5, 4); ax.set_ylim(-0.5, 5)
ax.set_aspect('equal'); ax.grid(True, alpha=0.3)
ax.set_title('Vector Addition', pad=12)

# Right: dot product and orthogonality
ax2 = axes[1]
a = np.array([1.0, 0.0])
b = np.array([0.0, 1.0])
c = np.array([1.0, 1.0]) / np.sqrt(2)

for vec, color, label in [(a, '#6366f1', 'a'), (b, '#2dd4bf', 'b'),
                          (c * 2, '#f97316', 'c (45°)')]:
    ax2.annotate('', xy=vec, xytext=[0,0],
                 arrowprops=dict(arrowstyle='->', color=color, lw=2))
    ax2.text(vec[0]*1.1, vec[1]*1.1, f'{label}\ndot={np.dot(a,vec):.2f}',
             color=color, fontsize=10)

ax2.set_xlim(-0.3, 1.8); ax2.set_ylim(-0.3, 1.8)
ax2.set_aspect('equal'); ax2.grid(True, alpha=0.3)
ax2.set_title('Dot Products vs. a=[1,0]', pad=12)

plt.tight_layout()
plt.show()

## Norms and normalization

The Euclidean norm $\|\mathbf{v}\| = \sqrt{v_1^2 + v_2^2}$ measures the length of a vector.
Dividing a vector by its norm produces a **unit vector** with direction but no magnitude.

In [ ]:
vectors_to_normalize = [
    np.array([3.0, 4.0]),
    np.array([1.0, 1.0]),
    np.array([5.0, 0.0]),
]

for v in vectors_to_normalize:
    norm = np.linalg.norm(v)
    unit = v / norm
    print(f'v={v}  ‖v‖={norm:.3f}  unit={unit.round(3)}  ‖unit‖={np.linalg.norm(unit):.4f}')

## Cosine similarity in practice

Cosine similarity is the dot product of **unit** vectors. It's used for word embeddings,
document similarity, and recommendation systems.

In [ ]:
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# Simulated word embeddings (toy example)
embeddings = {
    'king':   np.array([0.9, 0.8, 0.1, 0.0]),
    'queen':  np.array([0.8, 0.9, 0.2, 0.0]),
    'man':    np.array([0.7, 0.1, 0.0, 0.0]),
    'woman':  np.array([0.6, 0.2, 0.0, 0.0]),
    'apple':  np.array([0.0, 0.0, 0.8, 0.9]),
}

words = list(embeddings.keys())
for i, w1 in enumerate(words):
    for w2 in words[i+1:]:
        sim = cosine_similarity(embeddings[w1], embeddings[w2])
        if sim > 0.9:
            print(f'{w1:8s} ↔ {w2:8s}: similarity = {sim:.3f}')

## Projection — the dot product's geometric job

The dot product measures *how much of $\mathbf{b}$ points along $\mathbf{a}$*. Starting from $\mathbf{a}\cdot\mathbf{b} = \|\mathbf{a}\|\,\|\mathbf{b}\|\cos\theta$:

$$
\text{scalar proj} = \|\mathbf{b}\|\cos\theta = \frac{\mathbf{a}\cdot\mathbf{b}}{\|\mathbf{a}\|},
\qquad
\operatorname{proj}_{\mathbf{a}}\mathbf{b} = \frac{\mathbf{a}\cdot\mathbf{b}}{\mathbf{a}\cdot\mathbf{a}}\,\mathbf{a}.
$$

The residual $\mathbf{b} - \operatorname{proj}_{\mathbf{a}}\mathbf{b}$ is **orthogonal** to $\mathbf{a}$ — the property that makes least-squares regression and PCA work.

In [ ]:
a = np.array([3.0, 1.0])
b = np.array([1.0, 2.0])

# scalar projection = ||b|| cos(theta) = (a·b) / ||a||
scalar_proj = np.dot(a, b) / np.linalg.norm(a)
# vector projection = (a·b)/(a·a) * a   -- lands on the line through a
vector_proj = (np.dot(a, b) / np.dot(a, a)) * a
residual = b - vector_proj

print(f'scalar projection of b onto a : {scalar_proj:.3f}')
print(f'vector projection             : {vector_proj.round(3)}')
print(f'residual  b - proj            : {residual.round(3)}')

# Key fact used by least squares: the residual is ORTHOGONAL to a
print(f'residual · a = {np.dot(residual, a):.2e}  (=> 0, so residual ⟂ a)')

## Linear independence, span, and rank

Stack vectors as the **columns** of a matrix. The **rank** is the number of independent directions among them:

- rank = number of vectors  ⇒  **independent** — they span an $n$-dimensional subspace and form a basis for it.
- rank < number of vectors  ⇒  **dependent** — at least one is a linear combination of the others and adds no new direction.

In ML, a rank-deficient feature matrix means redundant features (perfect multicollinearity), which makes the normal equations singular and models unstable.

In [ ]:
def describe(vectors, name):
    M = np.column_stack(vectors)
    r = np.linalg.matrix_rank(M)
    n = M.shape[1]
    status = f'independent (spans R^{M.shape[0]})' if r == n else 'DEPENDENT'
    print(f'{name}: rank {r} of {n} columns -> {status}')

describe([np.array([1, 2]), np.array([2, 4])], 'a=[1,2], b=[2,4]')   # b = 2a -> dependent
describe([np.array([1, 2]), np.array([2, 1])], 'a=[1,2], c=[2,1]')   # independent

# With an independent basis we can express ANY target as a unique combination:
# solve M c = target  for the coefficients c
M = np.column_stack([np.array([1.0, 2.0]), np.array([2.0, 1.0])])
target = np.array([4.0, 5.0])
coeffs = np.linalg.solve(M, target)
print(f'\n[4,5] = {coeffs[0]:.2f}*[1,2] + {coeffs[1]:.2f}*[2,1]')
print('check:', (coeffs[0] * M[:, 0] + coeffs[1] * M[:, 1]).round(3))

## Vector visualization — arrows, dot product angle, and unit circle

The plot below shows three example vectors as arrows from the origin, the angle between **u** and **v** via the dot product, and a unit circle as a normalization reference.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

u = np.array([2.0, 1.0])
v = np.array([1.0, 3.0])
w = np.array([-1.0, 2.0])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── Left: three vectors as arrows ──────────────────────────────────────────
ax = axes[0]
vecs   = [u, v, w]
colors = ['#6366f1', '#2dd4bf', '#f97316']
labels = [r'$\mathbf{u}=[2,1]$', r'$\mathbf{v}=[1,3]$', r'$\mathbf{w}=[-1,2]$']

for vec, col, lbl in zip(vecs, colors, labels):
    ax.annotate('', xy=vec, xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color=col, lw=2.5))
    ax.text(vec[0] + 0.08, vec[1] + 0.08, lbl, color=col, fontsize=11)

# Dot-product angle arc between u and v
cos_theta = np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v))
angle_deg = np.degrees(np.arccos(np.clip(cos_theta, -1, 1)))
theta1 = np.degrees(np.arctan2(u[1], u[0]))
theta2 = np.degrees(np.arctan2(v[1], v[0]))
arc = mpatches.Arc((0, 0), 0.7, 0.7, angle=0,
                   theta1=min(theta1, theta2), theta2=max(theta1, theta2),
                   color='#f59e0b', lw=1.5)
ax.add_patch(arc)
mid_angle = np.radians((theta1 + theta2) / 2)
ax.text(0.45 * np.cos(mid_angle), 0.45 * np.sin(mid_angle),
        f'{angle_deg:.0f}°', color='#f59e0b', fontsize=10, ha='center')

ax.set_xlim(-1.8, 3.0); ax.set_ylim(-0.5, 3.8)
ax.set_aspect('equal'); ax.grid(True, alpha=0.3)
ax.axhline(0, color='#30344a', lw=0.8); ax.axvline(0, color='#30344a', lw=0.8)
ax.set_title('Three vectors + dot-product angle', pad=12)

# ── Right: unit circle + normalization ─────────────────────────────────────
ax2 = axes[1]
theta = np.linspace(0, 2 * np.pi, 300)
ax2.plot(np.cos(theta), np.sin(theta), color='#94a3b8', lw=1.2,
         linestyle='--', label='Unit circle')

for vec, col, lbl in zip(vecs, colors, labels):
    unit = vec / np.linalg.norm(vec)
    # original (faded)
    ax2.annotate('', xy=vec * 0.5, xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color=col, lw=1.5, alpha=0.3))
    # normalized (on the unit circle)
    ax2.annotate('', xy=unit, xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color=col, lw=2.5))
    ax2.text(unit[0] * 1.15, unit[1] * 1.15,
             lbl.replace('$', '').replace('\\mathbf', '').replace('{', '').replace('}', ''),
             color=col, fontsize=9, ha='center')

ax2.set_xlim(-1.6, 1.6); ax2.set_ylim(-1.6, 1.6)
ax2.set_aspect('equal'); ax2.grid(True, alpha=0.3)
ax2.axhline(0, color='#30344a', lw=0.8); ax2.axvline(0, color='#30344a', lw=0.8)
ax2.set_title('Normalized vectors on the unit circle', pad=12)
ax2.legend(fontsize=9, loc='lower right')

plt.suptitle('Vectors, Dot-Product Angle, and Normalization', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

---
## ✏️ Your turn

The cells below are **exercise scaffolds**: the concept is recapped, the code outline is set, and `# TODO(you)` marks what you fill in. Run the `assert` cell after each — it passes silently when your answer is right.

### Exercise 1 — Cosine similarity

Cosine similarity measures the *angle* between two vectors, ignoring their lengths:

$$\text{cos\_sim}(\mathbf{u}, \mathbf{v}) = \frac{\mathbf{u} \cdot \mathbf{v}}{\lVert \mathbf{u} \rVert \, \lVert \mathbf{v} \rVert}$$

It is +1 for parallel vectors, 0 for perpendicular ones, and −1 for opposite ones — the workhorse of embedding search. Implement it.

In [ ]:
def cosine_similarity(u, v):
    """Cosine of the angle between u and v."""
    u = np.asarray(u, dtype=float)
    v = np.asarray(v, dtype=float)

    # TODO(you): dot product of u and v (hint: np.dot)
    dot = ...

    # TODO(you): product of the two norms (hint: np.linalg.norm)
    norms = ...

    return dot / norms

In [ ]:
# Checks — run me
assert abs(cosine_similarity([1, 0], [0, 1])) < 1e-12, "perpendicular vectors -> 0"
assert abs(cosine_similarity([2, 1], [4, 2]) - 1) < 1e-12, "parallel vectors -> 1"
assert abs(cosine_similarity([1, 0], [-1, 0]) + 1) < 1e-12, "opposite vectors -> -1"
assert abs(cosine_similarity([1, 0], [1, 1]) - np.sqrt(2) / 2) < 1e-12, "45° apart -> cos(45°) = √2/2"
print("✅ Exercise 1 passed")

<details>
<summary>💡 Show solution</summary>

```python
def cosine_similarity(u, v):
    u = np.asarray(u, dtype=float)
    v = np.asarray(v, dtype=float)
    dot = np.dot(u, v)
    norms = np.linalg.norm(u) * np.linalg.norm(v)
    return dot / norms
```

</details>

### Exercise 2 — Vector projection

The projection of $\mathbf{v}$ onto $\mathbf{u}$ is the piece of $\mathbf{v}$ that points along $\mathbf{u}$:

$$\text{proj}_{\mathbf{u}}(\mathbf{v}) = \frac{\mathbf{v} \cdot \mathbf{u}}{\mathbf{u} \cdot \mathbf{u}} \, \mathbf{u}$$

What's left over, $\mathbf{v} - \text{proj}_{\mathbf{u}}(\mathbf{v})$, is always **orthogonal** to $\mathbf{u}$ — the checks verify exactly that.

In [ ]:
def project(v, u):
    """Vector projection of v onto u."""
    v = np.asarray(v, dtype=float)
    u = np.asarray(u, dtype=float)

    # TODO(you): scalar coefficient (v·u) / (u·u)
    coef = ...

    # TODO(you): scale u by that coefficient
    return ...

In [ ]:
# Checks — run me
assert np.allclose(project([3, 4], [1, 0]), [3, 0]), "projection onto the x-axis keeps only x"
assert np.allclose(project([2, 2], [0, 5]), [0, 2]), "projection onto the y-axis keeps only y"

v, u = np.array([3.0, 4.0]), np.array([1.0, 2.0])
assert abs(np.dot(v - project(v, u), u)) < 1e-12, "the residual must be orthogonal to u"
assert np.allclose(project(u, u), u), "projecting u onto itself returns u"
print("✅ Exercise 2 passed")

<details>
<summary>💡 Show solution</summary>

```python
def project(v, u):
    v = np.asarray(v, dtype=float)
    u = np.asarray(u, dtype=float)
    coef = np.dot(v, u) / np.dot(u, u)
    return coef * u
```

</details>